# CyberDiviner Gemma 3 1B QLoRA 微调

**目标**: 把 Gemma 3 1B 训练成 CyberDiviner 的格式稳定器和古风占卜文案生成器。

**运行要求**: Colab T4 GPU (免费即可)

**数据**: 78 条 SFT 样本 (oracle/tarot/liuyao/vision 四个功能)

**输出**: 合并后的 HF 模型 + GGUF Q4_K_M 文件 (用于 LiteRT-LM 转换)

## 0. 环境准备

⚠️ **运行前检查**:
1. 左侧菜单 → 运行时 → 更改运行时类型 → **T4 GPU**
2. 去 [HuggingFace](https://huggingface.co/google/gemma-3-1b-it) 接受 Gemma 许可协议
3. 左侧 🔑 图标 → 添加 `HF_TOKEN` (你的 HuggingFace access token)

In [ ]:
%%capture
!pip install unsloth protobuf==5.29.4 datasets
!pip install --no-deps --upgrade transformers>=4.51.0

In [ ]:
import torch
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"VRAM: {round(torch.cuda.get_device_properties(0).total_mem/1024**3,1)} GB")
print(f"Torch: {torch.__version__}")

In [ ]:
# Upload dataset
from google.colab import files
import os

print("请上传 cyberdiviner_sft_dataset.jsonl:")
uploaded = files.upload()
DATASET_PATH = list(uploaded.keys())[0]
print(f"✅ {DATASET_PATH} ({os.path.getsize(DATASET_PATH)} bytes)")

## 1. 加载模型 (Unsloth 4-bit)

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-3-1b-it",  # Unsloth 预量化版，Colab T4 可用
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,  # auto
)

print(f"✅ Model loaded")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"VRAM: {round(torch.cuda.memory_allocated()/1024**3, 2)} GB")

## 2. 配置 LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth 优化版
    random_state=42,
)

model.print_trainable_parameters()

## 3. 数据准备

In [ ]:
import json
from collections import Counter

# Load JSONL
raw_data = []
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            raw_data.append(json.loads(line))

print(f"Total samples: {len(raw_data)}")
features = Counter(item["feature"] for item in raw_data)
for feat, count in sorted(features.items()):
    print(f"  {feat}: {count}")

# Quality check
bad_words = ["请知会", "本地先知", "作为AI", "格式如下", "1到2句话", "px"]
bad_count = 0
for item in raw_data:
    for msg in item["messages"]:
        if msg["role"] == "assistant":
            for bw in bad_words:
                if bw in msg["content"]:
                    print(f"  ⚠️ '{bw}' found in: {msg['content'][:60]}...")
                    bad_count += 1
print(f"✅ Quality check: {bad_count} issues" if bad_count == 0 else f"❌ {bad_count} issues found")

In [ ]:
# Convert to Gemma chat format
def format_conversations(items):
    """Format messages using Gemma's chat template."""
    texts = []
    for item in items:
        messages = item["messages"]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return texts

all_texts = format_conversations(raw_data)
print(f"Formatted {len(all_texts)} samples")
print(f"\nSample (first 400 chars):")
print(all_texts[0][:400])

In [ ]:
import random
from datasets import Dataset

random.seed(42)

# Pair texts with features for stratified-ish split
paired = list(zip(all_texts, [item["feature"] for item in raw_data]))
random.shuffle(paired)

split_idx = int(len(paired) * 0.9)
train_pairs = paired[:split_idx]
val_pairs = paired[split_idx:]

train_dataset = Dataset.from_dict({"text": [t for t, _ in train_pairs]})
val_dataset = Dataset.from_dict({"text": [t for t, _ in val_pairs]})

val_features = Counter(f for _, f in val_pairs)
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")
print(f"Val features: {dict(val_features)}")

## 4. 训练

In [ ]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "./cyberdiviner-gemma3-1b-qlora"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,  # effective batch = 8
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    max_seq_length=max_seq_length,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    bf16=True,
    report_to="tensorboard",
    seed=42,
    dataset_text_field="text",
    dataset_num_proc=2,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

print("✅ Trainer configured")
print(f"  Epochs: 5, Batch: 2×4=8, LR: 2e-4")
print(f"  Max seq: {max_seq_length}, Warmup: 3%")
print(f"  VRAM: {round(torch.cuda.memory_allocated()/1024**3, 2)} GB")

In [ ]:
# Train!
import time
start = time.time()

train_result = trainer.train()

elapsed = time.time() - start
print(f"\n✅ Training complete in {elapsed/60:.1f} minutes")
print(f"Final train loss: {train_result.training_loss:.4f}")
print(f"Runtime: {train_result.metrics['train_runtime']:.0f}s")

# Save LoRA adapter
trainer.save_model(f"{OUTPUT_DIR}/lora_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
print(f"\n💾 LoRA adapter saved to {OUTPUT_DIR}/lora_adapter")

## 5. 推理测试

In [ ]:
# Enable inference mode
FastLanguageModel.for_inference(model)

test_prompts = [
    {"system": "你是CyberDiviner叩问天机。只输出[ 载入签文 ][ 逻辑解析 ][ 最终断语 ]。签文必须四句古风短句，不要祝福语，不要直接复述问题。",
     "user": "问题：下个月面试能成功吗？"},
    {"system": "你是CyberDiviner赛博塔罗。只输出塔罗解读、牌阵总论、逐牌详析、最终指引。中文成品，不要数字代号，不要提示词。",
     "user": "牌面：月亮（正位）。问题：感情走向。"},
    {"system": "你是CyberDiviner周易六爻。只输出[ 卦象解读 ][ 进退之策 ]。不得输出数字串、规则说明或提示词。",
     "user": "卦名：泰；上卦：坤；下卦：乾；动爻：五爻；问题：创业。"},
    {"system": "你是CyberDiviner视界摸骨。只输出面形总论、逐部位详析、运势总判。禁止px、比例、小数、字段名、英文标签。",
     "user": "面相特征：火形尖面，左右略偏；天庭窄；眉浓，眼锐有光；鼻尖，唇薄；地阁削。"},
]

bad_words = ["请知会", "本地先知", "作为AI", "格式如下", "1到2句话", "px"]
all_outputs = []

for i, prompt in enumerate(test_prompts):
    messages = [
        {"role": "system", "content": prompt["system"]},
        {"role": "user", "content": prompt["user"]},
    ]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            use_cache=True,
        )

    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    all_outputs.append(response)
    print(f"\n{'='*60}")
    print(f"Test {i+1}: {prompt['user'][:50]}")
    print(f"{'='*60}")
    print(response)

# Quality gate
print(f"\n{'='*60}")
print("Quality Gate:")
for bw in bad_words:
    found = any(bw in out for out in all_outputs)
    print(f"  '{bw}': {'❌' if found else '✅'}")

# Check for repetition
for i, out in enumerate(all_outputs):
    lines = [l.strip() for l in out.split('\n') if l.strip()]
    if len(lines) > len(set(lines)):
        print(f"  ⚠️ Test {i+1}: repetitive lines detected")

## 6. 合并 LoRA + 导出 GGUF

In [ ]:
# Save LoRA adapter (for Unsloth GGUF conversion)
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
print("✅ LoRA adapter saved")

In [ ]:
# Merge to 16-bit HF model
MERGED_DIR = "./cyberdiviner-gemma3-1b-merged"

# Re-load base model in 16-bit for clean merge
print("Loading base model in 16-bit for merge...")
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-3-1b-it",
    max_seq_length=max_seq_length,
    load_in_4bit=False,
    dtype=torch.float16,
)

# Load and merge LoRA
from peft import PeftModel
merged = PeftModel.from_pretrained(base_model, "lora_model")
merged = merged.merge_and_unload()

merged.save_pretrained(MERGED_DIR, safe_serialization=True)
base_tokenizer.save_pretrained(MERGED_DIR)

import os
total = sum(os.path.getsize(os.path.join(MERGED_DIR, f)) for f in os.listdir(MERGED_DIR))
print(f"✅ Merged model saved to {MERGED_DIR} ({total/1024**3:.2f} GB)")

In [ ]:
# Convert to GGUF Q4_K_M using Unsloth (built-in, no llama.cpp needed)
print("Converting to GGUF Q4_K_M...")

# Use Unsloth's built-in GGUF export
model.save_pretrained_gguf(
    "cyberdiviner_gguf",
    tokenizer,
    quantization_method="q4_k_m",
)

print("✅ GGUF conversion complete")

# List GGUF files
import glob
for f in glob.glob("cyberdiviner_gguf/*.gguf"):
    size_mb = os.path.getsize(f) / 1024**2
    print(f"  {f}: {size_mb:.1f} MB")

## 7. 下载产物

In [ ]:
from google.colab import files
import glob

# Download GGUF
gguf_files = glob.glob("cyberdiviner_gguf/*.gguf")
if gguf_files:
    gguf_path = gguf_files[0]
    print(f"Downloading {gguf_path}...")
    files.download(gguf_path)

# Also download LoRA adapter as backup
print("\nAlso available:")
print(f"  Merged HF model: {MERGED_DIR}/")
print(f"  LoRA adapter: lora_model/")

# Optionally zip the merged model
!zip -r cyberdiviner-gemma3-1b-merged.zip {MERGED_DIR}/
files.download("cyberdiviner-gemma3-1b-merged.zip")

## 8. 训练总结

### 产物
| 文件 | 用途 |
|------|------|
| `cyberdiviner_gguf/*.gguf` | GGUF Q4_K_M → LiteRT-LM 转 `.task` |
| `lora_model/` | LoRA adapter 备份 |
| `cyberdiviner-gemma3-1b-merged/` | 合并后完整 HF 模型 |

### 训练参数
- 基座: `unsloth/gemma-3-1b-it` (4-bit QLoRA)
- LoRA: r=16, alpha=32, dropout=0.05
- 数据: 78 样本, 90/10 拆分
- 训练: 5 epochs, lr=2e-4, batch=8, cosine schedule

### 下一步 (LiteRT-LM 转换)
```bash
# 1. 安装 Google AI Edge 转换工具
# 2. GGUF → .task 转换
# 3. adb push 到设备替换 offline_model/gemma3_1b_int4.task
# 4. adb logcat 验证 LiteRT-LM initialized
```

### 验收标准
- [ ] 格式通过率 ≥ 95%
- [ ] 重复循环率 = 0
- [ ] 英文泄漏率 = 0
- [ ] 禁词命中率 = 0
- [ ] 端侧生成 < 45s